# 🔐 Azure ML ↔ Key Vault Integration Test

This notebook validates integration between **Azure Machine Learning** and **Azure Key Vault** using managed identity.

## What This Tests

✅ **Azure ML workspace** can authenticate to **Key Vault** using managed identity  
✅ **RBAC permissions** are correctly configured  
✅ **Secrets can be read/written** from Azure ML compute  
✅ **Azure ML can use secrets** in training jobs and pipelines  

## Architecture

```
Azure ML Workspace (Managed Identity)
        ↓ RBAC: Key Vault Secrets User/Officer
Azure Key Vault
        ↓ Secrets stored securely
    api-keys, connection-strings, credentials
```

## Prerequisites

- Azure ML workspace with system-assigned managed identity
- Azure Key Vault with RBAC enabled
- Managed identity has `Key Vault Secrets User` or `Key Vault Secrets Officer` role

---

## 📦 Install Packages

In [ ]:
%pip install --upgrade azure-ai-ml azure-identity azure-keyvault-secrets azure-mgmt-keyvault requests --quiet

## 🔧 Configuration

These parameters can be set via Papermill when running as Azure ML job.

In [ ]:
# Parameters (can be overridden by Papermill)
import os

AZURE_SUBSCRIPTION_ID = os.getenv("AZURE_SUBSCRIPTION_ID", "<your-subscription-id>")
AZURE_RESOURCE_GROUP = os.getenv("AZURE_RESOURCE_GROUP", "<your-resource-group>")
AZUREML_WORKSPACE_NAME = os.getenv("AZUREML_WORKSPACE_NAME", "<your-aml-workspace>")
KEY_VAULT_NAME = os.getenv("KEY_VAULT_NAME", "<your-keyvault-name>")
KEY_VAULT_URL = f"https://{KEY_VAULT_NAME}.vault.azure.net"

print(f"Testing Azure ML → Key Vault Integration")
print(f"Subscription: {AZURE_SUBSCRIPTION_ID}")
print(f"Resource Group: {AZURE_RESOURCE_GROUP}")
print(f"AzureML Workspace: {AZUREML_WORKSPACE_NAME}")
print(f"Key Vault: {KEY_VAULT_NAME}")

## 🔑 Test 1: Authenticate to Azure

Use DefaultAzureCredential - automatically uses managed identity when running on Azure ML compute.

In [ ]:
from azure.identity import DefaultAzureCredential
from datetime import datetime
import logging

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

try:
    credential = DefaultAzureCredential(exclude_interactive_browser_credential=True)
    token = credential.get_token("https://management.azure.com/.default")
    
    print("✅ TEST 1 PASSED: Azure Authentication")
    print(f"   Token expires: {datetime.fromtimestamp(token.expires_on)}")
    logger.info("Azure authentication successful")
except Exception as e:
    print(f"❌ TEST 1 FAILED: {e}")
    raise

## 🤖 Test 2: Connect to Azure ML Workspace

In [ ]:
from azure.ai.ml import MLClient

try:
    ml_client = MLClient(
        credential=credential,
        subscription_id=AZURE_SUBSCRIPTION_ID,
        resource_group_name=AZURE_RESOURCE_GROUP,
        workspace_name=AZUREML_WORKSPACE_NAME
    )
    
    workspace = ml_client.workspaces.get(name=AZUREML_WORKSPACE_NAME)
    
    print("✅ TEST 2 PASSED: Azure ML Connection")
    print(f"   Workspace: {workspace.name}")
    print(f"   Location: {workspace.location}")
    
    if hasattr(workspace, 'identity') and workspace.identity:
        if hasattr(workspace.identity, 'principal_id'):
            print(f"   Managed Identity: {workspace.identity.principal_id}")
    
    logger.info("Azure ML connection validated")
except Exception as e:
    print(f"❌ TEST 2 FAILED: {e}")
    raise

## 🔐 Test 3: Connect to Key Vault

In [ ]:
from azure.keyvault.secrets import SecretClient
from azure.core.exceptions import HttpResponseError

try:
    kv_client = SecretClient(vault_url=KEY_VAULT_URL, credential=credential)
    
    # Test List permission
    try:
        secret_properties = list(kv_client.list_properties_of_secrets())
        print("✅ TEST 3 PASSED: Key Vault Connection")
        print(f"   Vault URL: {KEY_VAULT_URL}")
        print(f"   Secrets found: {len(secret_properties)}")
        print(f"   List permission: ✅")
    except HttpResponseError as e:
        if e.status_code == 403:
            print("✅ TEST 3 PASSED: Key Vault Connection (no List permission)")
            print(f"   Vault URL: {KEY_VAULT_URL}")
            print(f"   List permission: ❌ (not required for Get/Set)")
        else:
            raise
    
    logger.info("Key Vault connection validated")
except Exception as e:
    print(f"❌ TEST 3 FAILED: {e}")
    raise

## 📝 Test 4: Write Secret to Key Vault

Requires `Key Vault Secrets Officer` role.

In [ ]:
import uuid
from datetime import datetime

test_secret_name = "azureml-integration-test"
test_secret_value = f"test-{uuid.uuid4().hex[:8]}-{datetime.now().strftime('%Y%m%d%H%M%S')}"

try:
    secret = kv_client.set_secret(
        name=test_secret_name,
        value=test_secret_value,
        content_type="text/plain",
        tags={"CreatedBy": "AzureML_KeyVault_Test", "Purpose": "IntegrationTest"}
    )
    
    print("✅ TEST 4 PASSED: Write Secret")
    print(f"   Secret Name: {secret.name}")
    print(f"   Created: {secret.properties.created_on}")
    logger.info(f"Secret '{test_secret_name}' written successfully")
    
except HttpResponseError as e:
    if e.status_code == 403:
        print("⚠️  TEST 4 SKIPPED: No write permission (read-only role)")
        test_secret_name = None
    else:
        print(f"❌ TEST 4 FAILED: {e}")
        raise
except Exception as e:
    print(f"❌ TEST 4 FAILED: {e}")
    raise

## 📖 Test 5: Read Secret from Key Vault

Requires `Key Vault Secrets User` role (minimum).

In [ ]:
if test_secret_name:
    try:
        retrieved_secret = kv_client.get_secret(test_secret_name)
        
        print("✅ TEST 5 PASSED: Read Secret")
        print(f"   Secret Name: {retrieved_secret.name}")
        print(f"   Value matches: {retrieved_secret.value == test_secret_value}")
        logger.info("Secret retrieval validated")
        
    except Exception as e:
        print(f"❌ TEST 5 FAILED: {e}")
        raise
else:
    print("⏭️  TEST 5 SKIPPED: No test secret available")

## 📊 Test 6: Use Secret in Azure ML Context

Demonstrate accessing secrets from Azure ML datastores.

In [ ]:
try:
    datastores = list(ml_client.datastores.list())
    
    print("✅ TEST 6 PASSED: Azure ML Datastore Access")
    print(f"   Datastores: {len(datastores)}")
    
    for ds in datastores[:3]:  # Show first 3
        print(f"   • {ds.name} ({ds.type})")
    
    logger.info("Azure ML datastore access validated")
except Exception as e:
    print(f"❌ TEST 6 FAILED: {e}")
    raise

## 📋 Test Summary

In [ ]:
import json
from datetime import datetime

test_results = {
    "test_suite": "Azure ML → Key Vault Integration",
    "timestamp": datetime.now().isoformat(),
    "configuration": {
        "subscription_id": AZURE_SUBSCRIPTION_ID,
        "resource_group": AZURE_RESOURCE_GROUP,
        "azureml_workspace": AZUREML_WORKSPACE_NAME,
        "key_vault": KEY_VAULT_NAME
    },
    "tests": [
        {"name": "Azure Authentication", "status": "passed"},
        {"name": "Azure ML Connection", "status": "passed"},
        {"name": "Key Vault Connection", "status": "passed"},
        {"name": "Write Secret", "status": "passed" if test_secret_name else "skipped"},
        {"name": "Read Secret", "status": "passed" if test_secret_name else "skipped"},
        {"name": "Azure ML Datastore Access", "status": "passed"}
    ]
}

print("\n" + "=" * 80)
print("TEST RESULTS SUMMARY")
print("=" * 80)
print(json.dumps(test_results, indent=2))
print("=" * 80)

passed = sum(1 for t in test_results["tests"] if t["status"] == "passed")
total = len(test_results["tests"])
print(f"\n✅ {passed}/{total} tests passed")
print("\n🎉 Azure ML ↔ Key Vault integration validated successfully!")

## 🧹 Cleanup (Optional)

In [ ]:
# Uncomment to delete test secret
# if test_secret_name:
#     kv_client.begin_delete_secret(test_secret_name).wait()
#     print(f"✅ Deleted test secret: {test_secret_name}")

print("ℹ️  Cleanup skipped. Uncomment code above to delete test secrets.")